In [1]:
import os
import re
import xml.etree.cElementTree as ET
from cltk.stop.arabic.stopword_filter import stopwords_filter
from cltk.corpus.arabic.alphabet import LETTERS
from nltk.stem.isri import ISRIStemmer
stemmer = ISRIStemmer()
nums='۰-۹'
letters=''.join(LETTERS)
reg='[^0-9A-Za-z'+str(nums)+str(letters)+']+'

def preprocess(text):
    txtList=stopwords_filter(text)
    l=[]
    for piece in txtList:
        piece=re.sub(reg, ' ', piece)
        piece=piece.split(' ')
        for word in piece:
            word=stemmer.stem(word)
            word=stopwords_filter(word)
            if len(word)!= 0:
                l.append(word[0])
    return l

def create_invertedIndex(docsTokens):
    invIndex={}
    for d in range(0, len(docsTokens)):
        for token in docsTokens[d]:
            word=token[0]
            tag=token[1]
            if token[0] not in invIndex:
                invIndex[word]={d:[[1,tag]]}
            elif d not in invIndex[word]:
                invIndex[word][d] = [[1,tag]]
            else:
                l=invIndex[word][d]
                l=[x[1] for x in l]
                try:
                    index=l.index(tag)
                    invIndex[word][d][index][0] = invIndex[word][d][index][0] + 1
                except:
                    invIndex[word][d].append([1,tag])
                
    return invIndex


class docInfo:
    def __init__(self, docName,docLen):
        self.docName = docName
        self.docLen = docLen
    def __repr__(self):
        return __str__(self)
    def __str__(self):
        return "Name = "+str(self.docName)+"\nLength = "+str(self.docLen)
    
    
from nltk.tag import StanfordPOSTagger
from nltk import word_tokenize
jar = '/home/mosab/Desktop/IR/Ar/Project/stanford-postagger-full-2018-02-27/stanford-postagger.jar'
model = '/home/mosab/Desktop/IR/Ar/Project/stanford-postagger-full-2018-02-27/models/arabic.tagger'
pos_tagger = StanfordPOSTagger(model, jar, encoding='utf8')

documentsInfo=[]
documentsNames=[]
docsTokens = []
tree = ET.parse("/media/mosab/D Drive/lessons/PhD/Semester2/Information Retrieval/Ass2/Ar/Arabic Corpus/clean.txt.utf8")
root = tree.getroot()
for DOC in root.findall('DOC'):
    DOCNO = int(DOC.find('DOCNO').text.strip())
    TEXT= DOC.find('TEXT').text.strip()
    TEXT = pos_tagger.tag(word_tokenize(TEXT))
    docTokens=[]
    for token in TEXT:
        temp=[]
        wordTag = token[1].split('/')
        if(len(wordTag)<=1):
            wordTag = token[0].split('/')
        if(len(wordTag)<=1):
            continue
        l=preprocess(wordTag[0])
        if len(l)==0:
            continue
        temp.append(l[0])
        temp.append(wordTag[1])
        docTokens.append(temp)
    docsTokens.append(docTokens)    
    documentsNames.append(DOCNO)
    documentsInfo.append(docInfo(DOCNO, len(docTokens)))
    
    
print("==================================Loading Documents is Done==================================")





import pickle

with open("generated/documentsNames.txt", "wb") as fp:
    pickle.dump(documentsNames, fp)
    
with open("generated/documentsInfo.txt", "wb") as fp:
    pickle.dump(documentsInfo, fp)

'''
with open("generated/docsTokens.txt", "wb") as fp:
    pickle.dump(docsTokens, fp)
'''

invertedIndex=create_invertedIndex(docsTokens)
with open("generated/invertedIndex.txt", "wb") as fp:
    pickle.dump(invertedIndex, fp)

print("==================================Saving is Done=============================================")


==================================Loading Documents is Done==================================
==================================Saving is Done=============================================


In [2]:
print(documentsInfo[0].docName)
documentsInfo[0].docLen

2489


7

In [3]:
print(len(docsTokens))
print(docsTokens[0])

2730
[['زاد', 'VBD'], ['عاد', 'DTNN'], ['هدي', 'NN'], ['خير', 'NN'], ['عبد', 'DTNN'], ['جزء', 'DTNN'], ['اول', 'DTJJ']]


In [4]:
for (t,y) in invertedIndex.items():
    for (d,b) in y.items():
        if len(invertedIndex[t][d]) >1:
            print(t,end=' ')
            print(d,end=' ')
            print(invertedIndex[t][d])
    

اخك 2381 [[1, 'VBD'], [2, 'NN'], [1, 'NNP']]
يكر 664 [[1, 'VBP'], [1, 'NNP']]
يكر 313 [[1, 'VBP'], [1, 'NN']]
ربه 1446 [[1, 'NN'], [1, 'NNP']]
ربه 825 [[2, 'NNP'], [1, 'NN']]
ربه 1595 [[3, 'NNP'], [1, 'NN']]
ربه 848 [[1, 'NNP'], [1, 'NN']]
ربه 1691 [[1, 'NN'], [1, 'NNP']]
ربه 507 [[1, 'NNP'], [1, 'NN']]
ربه 253 [[1, 'NNP'], [1, 'NN']]
واد 640 [[1, 'NN'], [1, 'JJR'], [1, 'NNP']]
واد 1837 [[1, 'DTNN'], [1, 'NN']]
واد 563 [[2, 'DTNN'], [1, 'DTNNP']]
واد 532 [[1, 'NN'], [1, 'NNP']]
واد 1176 [[2, 'NNP'], [1, 'VBD']]
واد 1177 [[1, 'DTNN'], [1, 'NN']]
واد 1305 [[2, 'VBD'], [2, 'NNP']]
واد 414 [[1, 'NN'], [1, 'DTNN']]
واد 1338 [[1, 'DTNN'], [1, 'DTNNP']]
واد 309 [[1, 'NN'], [1, 'VN']]
واد 586 [[2, 'DTNN'], [1, 'VN']]
واد 162 [[1, 'DTNN'], [1, 'NN']]
واد 1239 [[2, 'DTNN'], [2, 'NN']]
واد 1333 [[1, 'VBD'], [1, 'DTNNP']]
واد 1704 [[1, 'JJ'], [1, 'NN']]
واد 1270 [[1, 'NNP'], [1, 'NN'], [1, 'DTNN']]
واد 1517 [[1, 'DTNN'], [1, 'JJ']]
واد 1535 [[1, 'JJ'], [1, 'NN']]
وذت 25 [[4, 'NN'], [1, 'NNP']]
خشو

ايا 936 [[1, 'VBD'], [4, 'NNP'], [1, 'NN']]
تطب 577 [[2, 'VBP'], [1, 'NN']]
تطب 2634 [[1, 'DTNN'], [1, 'VBP']]
تطب 171 [[1, 'DTNN'], [1, 'VBP']]
تطب 620 [[1, 'VBP'], [1, 'NN']]
خلع 1418 [[1, 'NNP'], [1, 'VBP']]
خلع 2190 [[1, 'DTNN'], [1, 'VBD']]
خلع 2192 [[2, 'VBD'], [2, 'NN']]
خلع 2193 [[1, 'DTNNP'], [4, 'DTNN']]
خلع 2194 [[1, 'DTJJ'], [2, 'VBD']]
خلع 2195 [[4, 'DTNN'], [1, 'VBD'], [1, 'JJ'], [1, 'NNP']]
خلع 2241 [[3, 'DTNN'], [1, 'NN']]
خلع 2617 [[2, 'DTNN'], [2, 'VBD'], [1, 'NNP']]
خلع 2276 [[1, 'NNP'], [1, 'NN']]
خلع 2611 [[2, 'DTNN'], [1, 'NN']]
خلع 2614 [[4, 'DTNN'], [1, 'NN']]
خلع 2616 [[1, 'NNP'], [1, 'DTNN'], [1, 'DTJJ']]
خلع 2553 [[1, 'VBD'], [1, 'NN']]
خلع 1018 [[1, 'VBD'], [1, 'NNP']]
خلع 1278 [[1, 'NN'], [1, 'NNP']]
لكك 1508 [[1, 'NNP'], [1, 'NN']]
هزل 2200 [[2, 'NNP'], [1, 'VBD'], [2, 'DTNN']]
هزل 2198 [[1, 'DTNN'], [1, 'NNP']]
هزل 2237 [[1, 'DTNN'], [1, 'NNP']]
عبس 2068 [[1, 'DTNN'], [1, 'NNP']]
عبس 2071 [[2, 'NNP'], [1, 'NN']]
عبس 2073 [[6, 'NNP'], [1, 'NN']]
عبس 1070 [

عدل 1758 [[1, 'NNP'], [1, 'NN']]
عدل 2275 [[1, 'DTNN'], [1, 'NN']]
عدل 1792 [[1, 'VN'], [1, 'NN']]
عدل 1285 [[4, 'NN'], [1, 'NNP'], [1, 'VBP'], [1, 'DTNN']]
عدل 274 [[5, 'VBP'], [3, 'NN']]
عدل 1819 [[1, 'NN'], [1, 'NNP']]
عدل 2340 [[1, 'NNS'], [1, 'NNP']]
عدل 1248 [[1, 'NNP'], [1, 'VBP']]
عدل 848 [[1, 'NNP'], [1, 'DTNN']]
عدل 1880 [[1, 'DTNN'], [3, 'NN']]
عدل 1889 [[1, 'NN'], [2, 'DTJJ'], [1, 'DTNN'], [1, 'JJ']]
عدل 1891 [[1, 'JJ'], [1, 'DTJJ']]
عدل 1902 [[5, 'JJ'], [1, 'DTJJ']]
عدل 2423 [[4, 'NN'], [3, 'DTNN'], [1, 'NNP']]
عدل 2445 [[1, 'DTNN'], [1, 'VBD']]
عدل 2487 [[2, 'NN'], [1, 'DTJJ']]
عدل 1695 [[2, 'NN'], [2, 'NNP'], [1, 'DTNN']]
عدل 2497 [[1, 'DTJJ'], [1, 'DTNN']]
عدل 2010 [[2, 'NN'], [1, 'VBP']]
عدل 472 [[3, 'VBP'], [2, 'NN']]
عدل 2471 [[1, 'NNP'], [1, 'DTNN']]
عدل 492 [[1, 'DTNN'], [1, 'NNP']]
عدل 1518 [[6, 'DTNN'], [1, 'NN']]
عدل 1903 [[2, 'NNP'], [1, 'NN']]
عدل 1022 [[1, 'DTNNS'], [1, 'NN']]
ل 570 [[1, 'NN'], [1, 'NNP']]
ل 581 [[1, 'NNP'], [1, 'VBD']]
ل 2153 [[1, 'NNP'], [1

فعل 235 [[1, 'VBP'], [3, 'NN']]
فعل 1265 [[1, 'VBP'], [1, 'NN']]
فعل 1267 [[3, 'VBD'], [1, 'VBP']]
فعل 253 [[5, 'NN'], [4, 'NNP'], [1, 'VBP']]
فعل 1285 [[1, 'NN'], [1, 'VBN']]
فعل 2313 [[2, 'NN'], [1, 'DTNN']]
فعل 1290 [[2, 'NNP'], [1, 'NN'], [1, 'VBD']]
فعل 267 [[1, 'VBP'], [1, 'NNP']]
فعل 270 [[1, 'NN'], [1, 'NNP']]
فعل 271 [[1, 'VBP'], [3, 'NN']]
فعل 1296 [[2, 'VBP'], [2, 'NN']]
فعل 275 [[1, 'DTNN'], [4, 'VBP'], [5, 'NN'], [1, 'VBD']]
فعل 281 [[1, 'VBD'], [1, 'VBP']]
فعل 1310 [[1, 'VBP'], [1, 'NN']]
فعل 2337 [[1, 'VBP'], [1, 'NN']]
فعل 1314 [[1, 'JJ'], [1, 'VBD'], [1, 'NN'], [1, 'VBP']]
فعل 2341 [[1, 'DTJJ'], [1, 'DTNN'], [1, 'NNP']]
فعل 294 [[1, 'NNP'], [2, 'NN'], [1, 'VBP']]
فعل 301 [[1, 'NN'], [1, 'VBP'], [1, 'JJ']]
فعل 302 [[7, 'NN'], [1, 'VBP']]
فعل 2354 [[1, 'NN'], [2, 'NNP']]
فعل 1247 [[1, 'VBD'], [1, 'NN']]
فعل 327 [[1, 'NN'], [1, 'VBP']]
فعل 2376 [[2, 'NN'], [1, 'VBP']]
فعل 333 [[1, 'VBD'], [1, 'VBP']]
فعل 1592 [[1, 'NNP'], [1, 'NN']]
فعل 1362 [[1, 'NNP'], [1, 'NN']]
فعل 33

بطن 1699 [[2, 'DTNN'], [2, 'NN']]
بطن 1711 [[1, 'NN'], [1, 'DTNN']]
بطن 1614 [[2, 'NN'], [5, 'DTNN']]
بطن 1526 [[2, 'NN'], [2, 'DTNN']]
بطن 248 [[1, 'NNP'], [1, 'NN']]
طيئ 1314 [[2, 'NN'], [1, 'NNP'], [1, 'JJ']]
طيئ 1323 [[1, 'NNP'], [1, 'NN']]
غرس 1254 [[1, 'NNP'], [1, 'NN']]
اتت 2568 [[1, 'NN'], [1, 'NNP']]
اتت 1976 [[1, 'VBD'], [2, 'NN']]
ونس 330 [[1, 'NNP'], [1, 'VBP']]
شمتو 744 [[1, 'NN'], [1, 'VBP']]
ترس 855 [[1, 'VBP'], [1, 'NNP']]
يثق 17 [[1, 'VN'], [1, 'JJ']]
يثق 1466 [[1, 'DTNN'], [1, 'VN']]
ؤثر 1302 [[6, 'VBP'], [1, 'VN'], [1, 'NN']]
ؤثر 2365 [[1, 'JJ'], [1, 'VBP']]
ؤثر 1662 [[1, 'JJ'], [2, 'NN'], [5, 'VBP'], [1, 'VN']]
تطع 473 [[2, 'DTNN'], [1, 'DTJJ'], [1, 'NN']]
تطع 295 [[1, 'DTNN'], [1, 'NN']]
تطع 366 [[2, 'DTNN'], [1, 'JJ'], [2, 'NNP'], [1, 'VBP']]
تطع 345 [[2, 'DTNN'], [2, 'VBP']]
يلء 2282 [[1, 'NN'], [2, 'DTJJ']]
يلء 2096 [[1, 'NN'], [1, 'DTNN']]
يلء 2297 [[1, 'NN'], [1, 'DTNN']]
يلء 2330 [[3, 'DTNN'], [1, 'NNP']]
يلء 2334 [[4, 'DTNN'], [1, 'NNP']]
فر 809 [[1, 'VBD'],

...

حفظ 294 [[1, 'NNP'], [1, 'NN']]
حفظ 299 [[1, 'NN'], [1, 'JJ'], [1, 'VBP']]
حفظ 301 [[1, 'VBP'], [1, 'NNP']]
حفظ 1840 [[3, 'NN'], [2, 'VBP']]
حفظ 305 [[2, 'DTNN'], [1, 'NN']]
حفظ 340 [[3, 'NN'], [1, 'VBD']]
حفظ 345 [[2, 'DTNN'], [1, 'DTNNP'], [1, 'NN']]
حفظ 2586 [[1, 'NN'], [1, 'DTNN']]
حفظ 359 [[1, 'JJ'], [1, 'DTNN'], [1, 'VBD']]
حفظ 360 [[1, 'NN'], [1, 'VBP'], [1, 'NNP']]
حفظ 1395 [[2, 'VBP'], [1, 'NN']]
حفظ 385 [[1, 'VBN'], [4, 'NN']]
حفظ 2447 [[3, 'NN'], [1, 'DTNN']]
حفظ 2448 [[1, 'NN'], [1, 'DTNN']]
حفظ 2457 [[3, 'VBP'], [1, 'NN'], [1, 'DTNN']]
حفظ 2467 [[1, 'VBP'], [1, 'NN']]
حفظ 1957 [[1, 'VBP'], [2, 'VBD'], [1, 'NN']]
حفظ 1447 [[1, 'VBP'], [1, 'NNP']]
حفظ 426 [[1, 'NN'], [1, 'VBP']]
حفظ 1481 [[1, 'NNP'], [1, 'JJ'], [2, 'NN']]
حفظ 1491 [[1, 'NN'], [1, 'NNP']]
حفظ 1493 [[6, 'DTNNP'], [2, 'NNP'], [1, 'NN'], [1, 'DTNN']]
حفظ 2641 [[1, 'DTNNP'], [1, 'DTNN']]
حفظ 485 [[2, 'VBP'], [1, 'JJ']]
حفظ 1514 [[2, 'NN'], [1, 'NNP']]
حفظ 2027 [[1, 'VBP'], [1, 'DTNN']]
زرق 1853 [[1, 'NN'], [1, 'D

حدث 315 [[2, 'NN'], [1, 'VBP']]
حدث 317 [[1, 'VBP'], [5, 'NN'], [2, 'DTNN']]
حدث 321 [[2, 'NN'], [1, 'DTNN']]
حدث 324 [[1, 'NNP'], [2, 'NN'], [6, 'DTNN']]
حدث 327 [[1, 'NN'], [1, 'DTNN']]
حدث 328 [[1, 'VBD'], [2, 'NN'], [1, 'NNP']]
حدث 329 [[3, 'NN'], [6, 'NNP'], [3, 'VBD'], [1, 'DTNN']]
حدث 330 [[4, 'NNP'], [2, 'NN'], [1, 'VBD'], [1, 'VBP']]
حدث 332 [[3, 'NN'], [1, 'VBD']]
حدث 2381 [[3, 'NNP'], [6, 'DTNN'], [1, 'JJ'], [1, 'NN']]
حدث 2445 [[3, 'NN'], [2, 'DTNN']]
حدث 2386 [[1, 'NN'], [2, 'DTNN'], [1, 'NNP']]
حدث 341 [[2, 'DTNN'], [1, 'NN']]
حدث 2390 [[1, 'DTNN'], [1, 'NNP']]
حدث 2391 [[2, 'NN'], [6, 'DTNN']]
حدث 2392 [[2, 'NN'], [4, 'DTNN']]
حدث 345 [[2, 'NNP'], [6, 'DTNN'], [10, 'NN'], [2, 'DTJJ'], [3, 'VBD']]
حدث 346 [[2, 'DTNN'], [5, 'NN'], [1, 'NNP']]
حدث 350 [[1, 'DTNN'], [2, 'NNP'], [1, 'NN']]
حدث 354 [[6, 'NN'], [1, 'DTNN'], [1, 'NNP']]
حدث 355 [[4, 'DTNN'], [6, 'NNP'], [1, 'DTNNP'], [1, 'DTJJ']]
حدث 359 [[6, 'NN'], [1, 'NNS'], [4, 'DTNN'], [3, 'NNP']]
حدث 364 [[3, 'NN'], [1, 'D

اية 1757 [[1, 'DTNN'], [1, 'NN']]
اية 2528 [[1, 'NN'], [3, 'DTNN']]
اية 234 [[1, 'DTNN'], [1, 'DTJJ']]
اية 2027 [[1, 'DTNN'], [1, 'DTJJ']]
اية 1265 [[2, 'DTNN'], [1, 'NNP']]
أهلل 540 [[1, 'NNP'], [1, 'NN']]
أهلل 522 [[1, 'NNP'], [2, 'NN']]
عبد 2 [[1, 'DTNN'], [1, 'DTNNP'], [1, 'VBP']]
عبد 3 [[3, 'NNP'], [2, 'NN'], [1, 'DTNN']]
عبد 5 [[3, 'NNP'], [1, 'DTNN'], [1, 'VBP']]
عبد 19 [[4, 'NNP'], [1, 'NN']]
عبد 2068 [[2, 'NNP'], [1, 'NN']]
عبد 25 [[2, 'NN'], [1, 'DTNNS'], [1, 'NNP']]
عبد 28 [[2, 'NN'], [2, 'NNP'], [1, 'NNS']]
عبد 32 [[2, 'NNP'], [1, 'NN']]
عبد 38 [[2, 'NN'], [1, 'NNP']]
عبد 39 [[1, 'DTNN'], [3, 'DTNNP']]
عبد 2396 [[4, 'NNP'], [1, 'VBD']]
عبد 47 [[1, 'NN'], [1, 'VBP']]
عبد 62 [[1, 'NNP'], [2, 'NN']]
عبد 2119 [[1, 'NN'], [1, 'NNP']]
عبد 72 [[1, 'NNP'], [4, 'NN']]
عبد 74 [[1, 'NN'], [1, 'VBD']]
عبد 80 [[3, 'NNP'], [1, 'VBD']]
عبد 2138 [[1, 'NN'], [1, 'DTNN']]
عبد 2144 [[1, 'DTNNP'], [1, 'DTNN']]
عبد 2064 [[2, 'NN'], [1, 'DTNNP']]
عبد 100 [[1, 'NN'], [1, 'NNP']]
عبد 2166 [[6, 'NN

ودع 759 [[1, 'NN'], [1, 'NNP']]
عمل 1028 [[2, 'DTNN'], [2, 'NNP']]
عمل 1030 [[1, 'NNP'], [1, 'NN']]
عمل 26 [[1, 'NN'], [1, 'DTNN']]
عمل 37 [[2, 'NNP'], [1, 'VBP'], [2, 'NN']]
عمل 1637 [[1, 'NN'], [1, 'JJ']]
عمل 1161 [[1, 'VN'], [1, 'NN'], [1, 'NNP']]
عمل 1880 [[4, 'NN'], [2, 'NNP']]
عمل 2199 [[2, 'DTNN'], [4, 'NN'], [1, 'DTNNP'], [1, 'NNP'], [1, 'VBP']]
عمل 158 [[2, 'NN'], [1, 'NNP']]
عمل 163 [[1, 'NN'], [1, 'JJ']]
عمل 1210 [[1, 'NNP'], [1, 'VBD']]
عمل 2241 [[2, 'NN'], [1, 'NNP']]
عمل 2255 [[1, 'JJ'], [1, 'DTNN'], [1, 'VBP']]
عمل 1240 [[1, 'VBD'], [1, 'NN'], [2, 'DTNN']]
عمل 1246 [[2, 'NNP'], [1, 'DTNNP'], [1, 'NN']]
عمل 506 [[1, 'DTNN'], [4, 'NN'], [1, 'NNP']]
عمل 39 [[1, 'DTNN'], [1, 'NN']]
عمل 244 [[4, 'NN'], [1, 'DTNN'], [1, 'JJ'], [1, 'NNP'], [1, 'VBN']]
عمل 2309 [[1, 'VBP'], [2, 'NN'], [1, 'DTNN'], [1, 'VBN']]
عمل 273 [[1, 'DTNN'], [1, 'NN']]
عمل 274 [[3, 'DTJJ'], [1, 'DTNN'], [1, 'NN']]
عمل 2327 [[1, 'NNP'], [1, 'DTNNP'], [1, 'VBP'], [1, 'DTNN']]
عمل 294 [[2, 'NNP'], [2, 'NN'], 

KeyboardInterrupt: 